# Facebook Network Analysis

This notebook analyzes the Facebook network data using NetworkX to examine various network properties.

In [ ]:
# Import necessary libraries
import networkx as nx
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import collections
from collections import Counter

# Set the style for the plots
plt.style.use('ggplot')
sns.set(style="whitegrid")

# For reproducibility
np.random.seed(42)

## Loading the Facebook Network Data

In [ ]:
# Load the Facebook network data
# The file format is an edge list where each line represents a friendship between two users
try:
    G = nx.read_edgelist('facebook_combined.txt', nodetype=int)
    print(f"Successfully loaded the Facebook network with {G.number_of_nodes()} nodes and {G.number_of_edges()} edges")
except FileNotFoundError:
    print("File not found. Please make sure 'facebook_combined.txt' is in the current directory.")

## Basic Network Properties

In [ ]:
# Calculate basic network properties
print(f"Number of nodes: {G.number_of_nodes()}")
print(f"Number of edges: {G.number_of_edges()}")
print(f"Is the graph directed? {G.is_directed()}")

# Check if the graph is connected
print(f"Is the graph connected? {nx.is_connected(G)}")

# Find the number of connected components
num_components = nx.number_connected_components(G)
print(f"Number of connected components: {num_components}")

# Find the largest connected component
largest_cc = max(nx.connected_components(G), key=len)
print(f"Size of the largest connected component: {len(largest_cc)}")

# Create a subgraph of the largest connected component
G_lcc = G.subgraph(largest_cc).copy()
print(f"Largest connected component has {G_lcc.number_of_nodes()} nodes and {G_lcc.number_of_edges()} edges")

## Degree Distribution Analysis

In [ ]:
# Calculate the degree distribution
degrees = [G.degree(n) for n in G.nodes()]
degree_counts = Counter(degrees)

# Create a dataframe for easier analysis
degree_df = pd.DataFrame.from_dict(degree_counts, orient='index').reset_index()
degree_df.columns = ['Degree', 'Count']
degree_df = degree_df.sort_values('Degree')

# Calculate degree statistics
avg_degree = np.mean(degrees)
median_degree = np.median(degrees)
max_degree = np.max(degrees)
min_degree = np.min(degrees)

print(f"Average degree: {avg_degree:.2f}")
print(f"Median degree: {median_degree:.0f}")
print(f"Maximum degree: {max_degree}")
print(f"Minimum degree: {min_degree}")

# Plot the degree distribution
plt.figure(figsize=(12, 6))

# Regular plot
plt.subplot(121)
plt.scatter(degree_df['Degree'], degree_df['Count'])
plt.xlabel('Degree (k)')
plt.ylabel('Number of nodes')
plt.title('Degree Distribution')
plt.grid(True)

# Log-log plot to check for power law
plt.subplot(122)
plt.loglog(degree_df['Degree'], degree_df['Count'], 'o')
plt.xlabel('Degree (k) - log scale')
plt.ylabel('Number of nodes - log scale')
plt.title('Degree Distribution (Log-Log Scale)')
plt.grid(True, which="both", ls="--")

plt.tight_layout()
plt.show()

## Path Length Analysis

In [ ]:
# Calculate the average shortest path length
# Note: We'll use the largest connected component as the full graph might have disconnected parts
avg_shortest_path = nx.average_shortest_path_length(G_lcc)
print(f"Average shortest path length: {avg_shortest_path:.4f}")

# Calculate the diameter (maximum shortest path length)
diameter = nx.diameter(G_lcc)
print(f"Network diameter: {diameter}")

# Calculate the distribution of path lengths
# For large graphs, we'll use a sample of node pairs to estimate
import random
sample_size = min(1000, G_lcc.number_of_nodes())
nodes = list(G_lcc.nodes())
sample_nodes = random.sample(nodes, sample_size)

path_lengths = []
for i in range(sample_size-1):
    for j in range(i+1, sample_size):
        source = sample_nodes[i]
        target = sample_nodes[j]
        path_length = nx.shortest_path_length(G_lcc, source=source, target=target)
        path_lengths.append(path_length)

# Plot the distribution of path lengths
path_counts = Counter(path_lengths)
path_df = pd.DataFrame.from_dict(path_counts, orient='index').reset_index()
path_df.columns = ['Path Length', 'Count']
path_df = path_df.sort_values('Path Length')

plt.figure(figsize=(10, 6))
plt.bar(path_df['Path Length'], path_df['Count'], alpha=0.7)
plt.xlabel('Shortest Path Length')
plt.ylabel('Frequency')
plt.title('Distribution of Shortest Path Lengths')
plt.grid(True, axis='y', alpha=0.3)
plt.show()

## Clustering Analysis

In [ ]:
# Calculate the global clustering coefficient (transitivity)
global_clustering = nx.transitivity(G)
print(f"Global clustering coefficient (transitivity): {global_clustering:.4f}")

# Calculate the average local clustering coefficient
avg_local_clustering = nx.average_clustering(G)
print(f"Average local clustering coefficient: {avg_local_clustering:.4f}")

# Calculate local clustering coefficients for each node
clustering_coeffs = nx.clustering(G)

# Create a dataframe for analysis
clustering_df = pd.DataFrame.from_dict(clustering_coeffs, orient='index').reset_index()
clustering_df.columns = ['Node', 'Clustering Coefficient']

# Plot histogram of clustering coefficients
plt.figure(figsize=(10, 6))
plt.hist(clustering_df['Clustering Coefficient'], bins=20, alpha=0.7)
plt.xlabel('Clustering Coefficient')
plt.ylabel('Number of Nodes')
plt.title('Distribution of Local Clustering Coefficients')
plt.grid(True, axis='y', alpha=0.3)
plt.axvline(avg_local_clustering, color='r', linestyle='dashed', linewidth=1, 
            label=f'Average: {avg_local_clustering:.3f}')
plt.legend()
plt.show()

In [ ]:
# Analyze the relationship between degree and clustering coefficient
degree_clustering = {}
for node in G.nodes():
    degree = G.degree(node)
    clustering = clustering_coeffs[node]
    if degree in degree_clustering:
        degree_clustering[degree].append(clustering)
    else:
        degree_clustering[degree] = [clustering]

# Calculate average clustering coefficient for each degree
avg_clustering_by_degree = {k: np.mean(v) for k, v in degree_clustering.items()}

# Convert to dataframe
dc_df = pd.DataFrame(list(avg_clustering_by_degree.items()), columns=['Degree', 'Avg Clustering'])
dc_df = dc_df.sort_values('Degree')

# Plot degree vs clustering coefficient
plt.figure(figsize=(12, 6))

# Regular plot
plt.subplot(121)
plt.scatter(dc_df['Degree'], dc_df['Avg Clustering'], alpha=0.6)
plt.xlabel('Degree')
plt.ylabel('Average Clustering Coefficient')
plt.title('Degree vs. Clustering Coefficient')
plt.grid(True, alpha=0.3)

# Log-log plot
plt.subplot(122)
plt.loglog(dc_df['Degree'], dc_df['Avg Clustering'], 'o', alpha=0.6)
plt.xlabel('Degree (log scale)')
plt.ylabel('Average Clustering Coefficient (log scale)')
plt.title('Degree vs. Clustering Coefficient (Log-Log Scale)')
plt.grid(True, which="both", ls="--", alpha=0.3)

plt.tight_layout()
plt.show()

## Centrality Analysis

In [ ]:
# For large networks, computing centrality measures can be time-consuming
# We'll work with the largest connected component for this analysis

# Calculate degree centrality
degree_centrality = nx.degree_centrality(G_lcc)

# Calculate betweenness centrality (this might take some time)
print("Computing betweenness centrality (this might take a while)...")
# Use approximate betweenness with sampling for large networks
k = min(1000, G_lcc.number_of_nodes())  # Number of sample nodes
betweenness_centrality = nx.betweenness_centrality(G_lcc, k=k, normalized=True, seed=42)

# Calculate closeness centrality
print("Computing closeness centrality...")
closeness_centrality = nx.closeness_centrality(G_lcc)

# Create a dataframe with centrality measures
centrality_df = pd.DataFrame({
    'Node': list(G_lcc.nodes()),
    'Degree Centrality': [degree_centrality[n] for n in G_lcc.nodes()],
    'Betweenness Centrality': [betweenness_centrality[n] for n in G_lcc.nodes()],
    'Closeness Centrality': [closeness_centrality[n] for n in G_lcc.nodes()]
})

# Display top 20 nodes by different centrality measures
print("\nTop 20 nodes by Degree Centrality:")
print(centrality_df.sort_values('Degree Centrality', ascending=False).head(20))

print("\nTop 20 nodes by Betweenness Centrality:")
print(centrality_df.sort_values('Betweenness Centrality', ascending=False).head(20))

print("\nTop 20 nodes by Closeness Centrality:")
print(centrality_df.sort_values('Closeness Centrality', ascending=False).head(20))

In [ ]:
# Compare the different centrality measures
plt.figure(figsize=(18, 6))

# Degree vs Betweenness
plt.subplot(131)
plt.scatter(centrality_df['Degree Centrality'], centrality_df['Betweenness Centrality'], alpha=0.5)
plt.xlabel('Degree Centrality')
plt.ylabel('Betweenness Centrality')
plt.title('Degree vs. Betweenness Centrality')
plt.grid(True, alpha=0.3)

# Degree vs Closeness
plt.subplot(132)
plt.scatter(centrality_df['Degree Centrality'], centrality_df['Closeness Centrality'], alpha=0.5)
plt.xlabel('Degree Centrality')
plt.ylabel('Closeness Centrality')
plt.title('Degree vs. Closeness Centrality')
plt.grid(True, alpha=0.3)

# Betweenness vs Closeness
plt.subplot(133)
plt.scatter(centrality_df['Betweenness Centrality'], centrality_df['Closeness Centrality'], alpha=0.5)
plt.xlabel('Betweenness Centrality')
plt.ylabel('Closeness Centrality')
plt.title('Betweenness vs. Closeness Centrality')
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Community Detection

In [ ]:
# Community detection using the Louvain algorithm
try:
    import community as community_louvain
    
    # Apply the Louvain algorithm
    print("Detecting communities using Louvain algorithm...")
    partition = community_louvain.best_partition(G_lcc)
    
    # Get the set of communities
    communities = set(partition.values())
    print(f"Number of communities detected: {len(communities)}")
    
    # Count nodes in each community
    community_sizes = Counter(partition.values())
    sizes_df = pd.DataFrame.from_dict(community_sizes, orient='index').reset_index()
    sizes_df.columns = ['Community', 'Size']
    sizes_df = sizes_df.sort_values('Size', ascending=False)
    
    print("\nTop 10 communities by size:")
    print(sizes_df.head(10))
    
    # Plot community size distribution
    plt.figure(figsize=(12, 6))
    
    plt.subplot(121)
    plt.bar(range(len(sizes_df)), sizes_df['Size'], alpha=0.7)
    plt.xlabel('Community Rank')
    plt.ylabel('Number of Nodes')
    plt.title('Community Size Distribution')
    plt.grid(True, axis='y', alpha=0.3)
    
    plt.subplot(122)
    plt.loglog(range(1, len(sizes_df) + 1), sizes_df['Size'], 'o-', alpha=0.7)
    plt.xlabel('Community Rank (log scale)')
    plt.ylabel('Number of Nodes (log scale)')
    plt.title('Community Size Distribution (Log-Log Scale)')
    plt.grid(True, which='both', ls='--', alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
except ImportError:
    print("The 'community' package is not installed. Install it using:")
    print("pip install python-louvain")

## Small World Properties

In [ ]:
# Test for small-world properties
# A network is a small-world network if:
# 1. It has a similar average path length to an equivalent random graph
# 2. It has a much higher clustering coefficient than an equivalent random graph

# Create a random graph with the same number of nodes and average degree
n = G_lcc.number_of_nodes()
m = G_lcc.number_of_edges()
p = 2 * m / (n * (n - 1))  # Probability of edge existence
random_G = nx.erdos_renyi_graph(n, p, seed=42)

# Calculate clustering coefficients
clustering_actual = nx.average_clustering(G_lcc)
clustering_random = nx.average_clustering(random_G)

# Calculate average path lengths
path_length_actual = nx.average_shortest_path_length(G_lcc)
path_length_random = nx.average_shortest_path_length(random_G)

print(f"Actual Network - Average Clustering: {clustering_actual:.4f}")
print(f"Random Network - Average Clustering: {clustering_random:.4f}")
print(f"Clustering Ratio (Actual/Random): {clustering_actual/clustering_random:.2f}x")

print(f"\nActual Network - Average Path Length: {path_length_actual:.4f}")
print(f"Random Network - Average Path Length: {path_length_random:.4f}")
print(f"Path Length Ratio (Actual/Random): {path_length_actual/path_length_random:.2f}x")

# Check sigma for small-worldness
sigma = (clustering_actual / clustering_random) / (path_length_actual / path_length_random)
print(f"\nSmall-worldness coefficient (σ): {sigma:.4f}")
print("A network is considered small-world if σ > 1")

## Network Visualization

In [ ]:
# For large networks, visualization of the entire network might be messy
# We'll visualize a subgraph with the top nodes by degree centrality

# Get top 50 nodes by degree centrality
top_nodes = centrality_df.sort_values('Degree Centrality', ascending=False).head(50)['Node'].tolist()
subgraph = G_lcc.subgraph(top_nodes)

plt.figure(figsize=(12, 12))

# Position nodes using force-directed layout
pos = nx.spring_layout(subgraph, seed=42)

# Get node degrees for sizing
degrees = dict(subgraph.degree())
node_sizes = [v * 100 for v in degrees.values()]

# Draw the network
nx.draw_networkx_nodes(subgraph, pos, node_size=node_sizes, alpha=0.7,
                       node_color=list(degrees.values()), cmap=plt.cm.viridis)
nx.draw_networkx_edges(subgraph, pos, alpha=0.2)
nx.draw_networkx_labels(subgraph, pos, font_size=8, font_color='black')

plt.title('Facebook Network - Top 50 Nodes by Degree Centrality')
plt.colorbar(plt.cm.ScalarMappable(cmap=plt.cm.viridis), 
             label='Node Degree')
plt.axis('off')
plt.tight_layout()
plt.show()

## Summary of Network Analysis

In [ ]:
# Create a summary of the network analysis
print("Facebook Network Analysis Summary")
print("=================================")
print(f"Number of nodes: {G.number_of_nodes()}")
print(f"Number of edges: {G.number_of_edges()}")
print(f"Average degree: {avg_degree:.2f}")
print(f"Network density: {nx.density(G):.6f}")
print(f"Global clustering coefficient: {global_clustering:.4f}")
print(f"Average local clustering coefficient: {avg_local_clustering:.4f}")
print(f"Average shortest path length: {path_length_actual:.4f}")
print(f"Network diameter: {diameter}")
print(f"Small-worldness coefficient (σ): {sigma:.4f}")
print(f"Number of connected components: {num_components}")
print(f"Size of the largest connected component: {len(largest_cc)} nodes ({len(largest_cc)/G.number_of_nodes()*100:.2f}% of the network)")

try:
    print(f"Number of communities detected: {len(communities)}")
except NameError:
    print("Community detection was not performed")